# RQ3, Part 2: Real Model Training (with the leakage fix)

Reproduces the real, documented leakage catch: fields populated only for Part I.A (never Part I.B) are excluded, with an automated, upfront check that flags any real feature with a large population-rate gap between classes before training.

**Requires `pcaob_deficiencies_raw.csv` from Part 1.**

In [1]:
!pip install -q pandas numpy scikit-learn statsmodels || pip install -q pandas numpy scikit-learn statsmodels --break-system-packages

In [2]:
"""
RQ3 - PART 2: Real Model Training (with the established leakage fix)
================================================================================
Reproduces the real, documented leakage catch from this project: an initial
100% accuracy was traced to fields populated ONLY for Part I.A records
(never for Part I.B) -- meaning the model was reading the label directly
off a field's mere presence, not learning anything real. "Audit Area" and
"Finding Count" are only ever populated for Part I.A in PCAOB's real data
structure (Part I.B records genuinely lack these fields), so both are
excluded here as real, disclosed leakage sources.

FIXED: a second real bug was found and reproduced before this fix -- the
original REAL_FEATURES list included "Paragraph of the Auditing Standard"
and "Firm-Identified Risk Assessment", which don't just have a leakage-
level population gap, they are ENTIRELY ABSENT from Part I.B's real data
structure (confirmed by directly fetching and comparing live PCAOB JSON
for both parts). Including them meant every real Part I.B row was dropped
during cleaning, leaving only one class and making training impossible --
this is what caused "only one severity class remains" errors. Fixed by
using only fields confirmed present in both real datasets.

Real features used: Auditing Standard, Inspection Type, Country,
Global Network, Inspection Year -- confirmed genuinely populated for both
real severity classes.
"""
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                               f1_score, roc_auc_score)
from statsmodels.stats.contingency_tables import mcnemar

# Real, disclosed leakage/structural-absence fields -- present only for
# Part I.A, either due to real leakage risk or because Part I.B's real
# data structure never collected them at all (confirmed via live fetch).
LEAKAGE_FIELDS = ["Audit Area", "Finding Count", "Paragraph of the Auditing Standard",
                   "Firm-Identified Risk Assessment", "Issuer Reference Key",
                   "Firm played a role but was not the lead auditor",
                   "Audits Affected by the Deficiencies Identified in Part I.A",
                   "Classification of Audits with Part I.A Deficiencies",
                   "Description in the Firm's Inspection Report"]

# FIXED: only fields confirmed present in BOTH real datasets
REAL_FEATURES = ["Auditing Standard", "Inspection Type", "Country",
                  "Global Network", "Inspection Year"]


def clean_empty_strings(df: pd.DataFrame) -> pd.DataFrame:
    """Real, necessary fix: PCAOB's real data uses empty strings, not NaN,
    for genuinely missing values -- pandas .notna() treats "" as populated,
    which silently breaks the leakage check and dropna() otherwise."""
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].replace("", np.nan)
    # Also drop fully-blank rows -- real, malformed artifacts present in
    # PCAOB's real JSON export (confirmed present in live data)
    df = df.dropna(how="all")
    return df


def verify_no_leakage(df: pd.DataFrame):
    """Real, upfront check: confirm the chosen real features are actually
    populated for both classes, not just one -- catches the same class of
    bug that caused the original 100% accuracy result."""
    print("Real leakage check (feature population rate by class):")
    for col in REAL_FEATURES:
        pct_1a = df[df.severity == 1][col].notna().mean() * 100
        pct_1b = df[df.severity == 0][col].notna().mean() * 100
        flag = " <<< POSSIBLE LEAKAGE" if abs(pct_1a - pct_1b) > 50 else ""
        print(f"  {col}: Part I.A={pct_1a:.1f}% populated, Part I.B={pct_1b:.1f}% populated{flag}")


def run_model_training():
    df = pd.read_csv("pcaob_deficiencies_raw.csv")
    df = clean_empty_strings(df)
    print(f"Real N = {len(df)} ({(df.severity==1).sum()} Part I.A, {(df.severity==0).sum()} Part I.B)")

    verify_no_leakage(df)

    df = df.dropna(subset=REAL_FEATURES + ["severity"])
    print(f"Real N after dropping missing values: {len(df)} "
          f"({(df.severity==1).sum()} Part I.A, {(df.severity==0).sum()} Part I.B)")

    if df["severity"].nunique() < 2:
        raise SystemExit("Real error: only one severity class remains after cleaning. "
                          "Check REAL_FEATURES against the real data's actual columns.")

    X_raw = df[REAL_FEATURES].astype(str)
    y = df["severity"].values

    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    X = encoder.fit_transform(X_raw)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    rf = RandomForestClassifier(n_estimators=300, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_proba = rf.predict_proba(X_test)[:, 1]

    logreg = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
    logreg.fit(X_train, y_train)
    lr_pred = logreg.predict(X_test)
    lr_proba = logreg.predict_proba(X_test)[:, 1]

    maj_pred = np.ones_like(y_test) if y_test.mean() > 0.5 else np.zeros_like(y_test)

    print("\n=== REAL RESULTS ===")
    for name, pred, proba in [("Random Forest", rf_pred, rf_proba), ("Logistic Regression", lr_pred, lr_proba)]:
        print(f"\n{name}:")
        print(f"  Accuracy:  {accuracy_score(y_test, pred):.3f}")
        print(f"  Precision: {precision_score(y_test, pred):.3f}")
        print(f"  Recall:    {recall_score(y_test, pred):.3f}")
        print(f"  F1:        {f1_score(y_test, pred):.3f}")
        print(f"  AUC:       {roc_auc_score(y_test, proba):.3f}")
    print(f"\nMajority baseline accuracy: {accuracy_score(y_test, maj_pred):.3f}")

    # Real McNemar's test comparing the two models
    both_correct = np.sum((rf_pred == y_test) & (lr_pred == y_test))
    rf_only = np.sum((rf_pred == y_test) & (lr_pred != y_test))
    lr_only = np.sum((rf_pred != y_test) & (lr_pred == y_test))
    both_wrong = np.sum((rf_pred != y_test) & (lr_pred != y_test))
    table = [[both_correct, rf_only], [lr_only, both_wrong]]
    result = mcnemar(table, exact=False, correction=True)
    print(f"\nReal McNemar's test (RF vs LogReg): chi-square={result.statistic:.3f}, p={result.pvalue:.4f}")


if __name__ == "__main__":
    run_model_training()


Real N = 17077 (14043 Part I.A, 3034 Part I.B)
Real leakage check (feature population rate by class):
  Auditing Standard: Part I.A=100.0% populated, Part I.B=99.7% populated
  Inspection Type: Part I.A=100.0% populated, Part I.B=99.7% populated
  Country: Part I.A=100.0% populated, Part I.B=99.7% populated
  Global Network: Part I.A=47.9% populated, Part I.B=24.9% populated
  Inspection Year: Part I.A=100.0% populated, Part I.B=99.7% populated
Real N after dropping missing values: 7489 (6733 Part I.A, 756 Part I.B)

=== REAL RESULTS ===

Random Forest:
  Accuracy:  0.962
  Precision: 0.990
  Recall:    0.967
  F1:        0.979
  AUC:       0.983

Logistic Regression:
  Accuracy:  0.971
  Precision: 0.990
  Recall:    0.978
  F1:        0.984
  AUC:       0.988

Majority baseline accuracy: 0.899

Real McNemar's test (RF vs LogReg): chi-square=3.250, p=0.0714
